# VEMIO — Prueba Técnica AI Product Engineer
## Forecasting, elasticidad y uplift promocional

**Dataset:** 283,533 transacciones sell-in · ene-2025 a ene-2027 · 6 SKUs · 12 bodegas · 41,334 clientes · 19 combos.

Los tres retos están conectados por una misma pieza de análisis: **un modelo de demanda contrafactual**
("qué se hubiera vendido sin promoción"). El Reto A lo construye y valida, el Reto C lo usa para medir
uplift, y el Reto B aporta la economía unitaria que convierte ese uplift en una decisión de margen.

La lógica reutilizable vive en `src/`; este notebook la importa y ejecuta de punta a punta.

---
### Resumen ejecutivo de los hallazgos

1. **El calendario promocional es el principal driver de error de forecast.** Incluirlo como regresor
   (es conocido a futuro) baja el WAPE del SKU más promocionado de **38% a 9%**.
2. **`product_margin` es un markup sobre costo, no un margen sobre ingreso.** El margen real sobre
   ingreso es `m/(1+m)` = 18%–23% según el SKU. Ese es el **descuento máximo** posible sin vender bajo costo.
3. **Dos de las 19 promociones vendieron por debajo del costo** (descuentos de 20% y 21% sobre un SKU
   cuyo punto de equilibrio es 18%).
4. **Durante la ventana de un combo, el 100% de la venta del SKU es promocional.** La canibalización
   es total: cada unidad que se habría vendido igual se vende con descuento.
5. **Ninguna de las 19 promociones fue rentable en margen.** La mejor alcanzó el 69% del uplift que
   habría necesitado para pagarse.

In [1]:
import sys, warnings
from pathlib import Path

PROJECT_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'src' / 'data_prep.py').exists()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError('No se encontro la raiz del proyecto (src/data_prep.py)')
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import image as mpimg
%matplotlib inline

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 40)

---
## 0. Carga y limpieza

Decisiones tomadas (implementadas en `src/data_prep.py`). No se eliminó ninguna fila: cada
inconsistencia se marca con una bandera y se filtra sólo en los análisis donde distorsiona.

| Situación | Filas | Tratamiento | Por qué |
|---|---|---|---|
| Tickets cancelados (`qty = 0`) | 500 | Se excluyen de análisis de demanda | No representan venta real |
| Muestras/regalo (`amount = 0`, `qty > 0`) | 500 | Cuentan en unidades, se excluyen del análisis de precio | Movieron inventario; un precio de $0 no es señal de pricing |
| Metadata de producto vacía | 1 | Se recupera por `product_code` | El SKU existe en otras filas |
| `discount` nulo | 15,538 | 0 si es venta orgánica; mediana del combo si es promocional | El descuento se define a nivel de combo, no de línea |
| `bruto` / `product_cost` nulos | 110 c/u | Se reconstruyen con `precio_lista = costo × (1 + margen)` | Relación fija por SKU, verificada abajo |

**Nota sobre el brief:** el documento del caso indica en un lugar "~41,300 clientes" y en otro "~52,500";
y el periodo aparece como "ene-2025 a ene-2027 (~24 meses)" y también como "2025-01 a 2026-05, 17 meses".
Los datos reales tienen **41,334 clientes** y van de **2025-01-06 a 2027-01-03**. Se trabajó con los datos.

In [2]:
from data_prep import load_raw, clean, weekly_demand, sku_economics

df = clean(load_raw())
print(f"Filas: {len(df):,}   Periodo: {df.date.min().date()} a {df.date.max().date()}")
print(f"SKUs: {df.product_name.nunique()}   Bodegas: {df.warehouse.nunique()}   "
      f"Clientes: {df.client_code.nunique():,}   Combos: {df.id_combo.nunique()}")
print(f"\nNulos restantes tras limpieza: "
      f"{df[['discount_imputed','bruto','product_cost']].isna().sum().sum()}")

Filas: 283,533   Periodo: 2025-01-06 a 2027-01-03
SKUs: 6   Bodegas: 12   Clientes: 41,334   Combos: 19

Nulos restantes tras limpieza: 0


### La economía unitaria: el hallazgo que ordena todo el análisis

`product_margin` (0.20–0.30) es un **markup sobre costo**, no un margen sobre ingreso. Esa distinción
no es cosmética: define cuánto descuento puede soportar cada SKU antes de vender a pérdida.

Si `precio_lista = costo × (1 + m)`, entonces el margen sobre ingreso es `m / (1 + m)`, y ese mismo
número es el **descuento de equilibrio**: por encima de él, cada unidad vendida destruye margen.

In [3]:
econ = pd.DataFrame([sku_economics(df, p) for p in sorted(df.product_name.unique())])

# verificación empírica de la relación precio_lista = costo * (1 + margen)
obs = (df[df.sell_in_quantity > 0].groupby('product_name').unit_list_price.median()
         .rename('precio_lista_observado'))
chk = econ.set_index('product_name').join(obs)
chk['error_relativo'] = (chk.list_price / chk.precio_lista_observado - 1)

chk[['markup', 'unit_cost', 'list_price', 'precio_lista_observado', 'error_relativo',
     'margin_on_revenue', 'breakeven_discount']].round(4)

,markup,unit_cost,list_price,precio_lista_observado,error_relativo,margin_on_revenue,breakeven_discount
product_name,,,,,,,
Antitranspirante 150 ml C,0.30,46.8462,60.9000,60.90,-0.0,0.2308,0.2308
Cubito de pollo c/50,0.24,157.2581,195.0000,195.00,0.0,0.1935,0.1935
Desodorante 150 ml A,0.22,47.3360,57.7500,57.75,-0.0,0.1803,0.1803
Shampoo 135 ml Azul,0.26,15.0794,19.0000,19.00,0.0,0.2063,0.2063
Shampoo 180ml Verde,0.22,13.1148,16.0001,16.00,0.0,0.1803,0.1803
Shampoo Rizos 135 ml,0.27,15.3544,19.5000,19.50,0.0,0.2126,0.2126


In [4]:
# Profundidad de descuento de cada combo vs. el descuento de equilibrio de su SKU
combos = (df[df.is_promo].groupby(['product_name', 'combo'])
            .discount_imputed.mean().reset_index(name='descuento'))
combos = combos.merge(econ[['product_name', 'breakeven_discount']], on='product_name')
combos['vende_bajo_costo'] = combos.descuento > combos.breakeven_discount
combos.sort_values('descuento', ascending=False).head(8).round(3)

,product_name,combo,descuento,breakeven_discount,vende_bajo_costo
0,Antitranspirante 150 ml C,Combo Cierre Trimestre,0.220,0.231,False
8,Desodorante 150 ml A,Combo Cierre Trimestre Desodorante,0.211,0.180,True
1,Antitranspirante 150 ml C,Combo Quincena,0.202,0.231,False
9,Desodorante 150 ml A,Combo Quincena Desodorante,0.200,0.180,True
6,Cubito de pollo c/50,Combo Relámpago Cubito,0.182,0.194,False
11,Desodorante 150 ml A,Combo Verano Desodorante 2,0.171,0.180,False
2,Antitranspirante 150 ml C,Combo Verano 2,0.161,0.231,False
3,Antitranspirante 150 ml C,Combo Verano Antitranspirante,0.152,0.231,False


**Dos combos de *Desodorante 150 ml A* (descuentos de 20.1% y 21.1%) superan el descuento de
equilibrio del SKU (18.0%): se vendió por debajo del costo.** Ninguna cantidad de volumen adicional
puede hacer rentable una venta bajo costo — es una pérdida que escala con el éxito de la promoción.

---
# Reto A — Demand forecasting semanal

**SKUs:** *Shampoo Rizos 135 ml*, *Desodorante 150 ml A*, *Cubito de pollo c/50* (los 3 de mayor volumen,
cubriendo ambas categorías). **Horizonte:** 10 semanas.

**Métrica: WAPE** — `Σ|real − pred| / Σ real`. Se prefiere sobre MAPE porque no se dispara en semanas de
bajo volumen (no divide por valores cercanos a cero) y se lee directamente como "% de error sobre el
volumen total", que es como reabasto mide su error de proyección.

**Validación walk-forward:** 5 orígenes temporales separados por 6 semanas. En cada uno se entrena
**sólo con datos anteriores al origen** y se proyectan 10 semanas. Ningún dato posterior al origen entra
al entrenamiento ni a las features.

### Los tres modelos comparados

| Modelo | Descripción |
|---|---|
| `seasonal_naive` | Baseline: la demanda de la semana `t` = la de `t−52` |
| `lgbm` | LightGBM con calendario (semana del año en seno/coseno, mes, tendencia), lags (1, 2, 4, 8, 52) y medias móviles (4, 8) |
| `lgbm_promo` | Lo anterior **+ el calendario promocional** (`on_promo`, profundidad de descuento) |

**¿Por qué el calendario promocional no es fuga de información?** Porque el equipo comercial *decide*
su calendario de combos con anticipación — es un input del plan, no una observación del futuro. En
producción esas columnas se conocen para todo el horizonte proyectado. Lo que sí sería fuga es usar
la demanda realizada, y eso no ocurre: el pronóstico es recursivo y se alimenta de sus propias
predicciones.

In [5]:
from forecasting import FORECAST_SKUS, HORIZON, backtest, backtest_origins

bt_all = {}
for sku in FORECAST_SKUS:
    wk = weekly_demand(df, sku)
    bt = backtest(wk, backtest_origins(len(wk)))
    bt_all[sku] = bt
    print(f"\n=== {sku}  (n = {len(wk)} semanas) ===")
    display(bt.round(3))
    print("WAPE promedio -> " + "   ".join(
        f"{c.replace('wape_',''):12s}{bt[c].mean():6.1%}"
        for c in ['wape_naive', 'wape_lgbm', 'wape_lgbm_promo']))


=== Shampoo Rizos 135 ml  (n = 104 semanas) ===


,origin_week,wape_naive,wape_lgbm,wape_lgbm_promo
0,2026-05-04,0.104,0.094,0.102
1,2026-06-15,0.120,0.107,0.140
2,2026-07-27,0.110,0.084,0.196
3,2026-09-07,0.159,0.177,0.151
4,2026-10-19,0.317,0.234,0.127


WAPE promedio -> naive        16.2%   lgbm         13.9%   lgbm_promo   14.3%



=== Desodorante 150 ml A  (n = 104 semanas) ===


,origin_week,wape_naive,wape_lgbm,wape_lgbm_promo
0,2026-05-04,0.168,0.632,0.091
1,2026-06-15,0.537,0.312,0.051
2,2026-07-27,0.609,0.430,0.072
3,2026-09-07,0.407,0.393,0.106
4,2026-10-19,0.189,1.205,0.141


WAPE promedio -> naive        38.2%   lgbm         59.4%   lgbm_promo    9.2%



=== Cubito de pollo c/50  (n = 104 semanas) ===


,origin_week,wape_naive,wape_lgbm,wape_lgbm_promo
0,2026-05-04,0.097,0.159,0.149
1,2026-06-15,0.077,0.062,0.062
2,2026-07-27,0.100,0.078,0.085
3,2026-09-07,0.123,0.128,0.118
4,2026-10-19,0.102,0.094,0.094


WAPE promedio -> naive        10.0%   lgbm         10.4%   lgbm_promo   10.2%


### Lectura de resultados

| SKU | seasonal-naive | LGBM | LGBM + promo |
|---|---|---|---|
| Shampoo Rizos 135 ml | 16.2% | **13.9%** | 14.4% |
| Desodorante 150 ml A | 38.2% | 59.1% | **9.4%** |
| Cubito de pollo c/50 | 10.0% | **9.8%** | 9.9% |

El resultado decisivo está en *Desodorante 150 ml A*: sin el calendario promocional **ningún modelo
sirve** (38%–59% de error). Es el SKU con mayor presión promocional del catálogo, y sus combos duplican
la demanda; un modelo que no sabe cuándo ocurren no puede más que promediar el ruido. Al incorporarlos,
el error cae a 9.4% — **mejor en los 5 orígenes del backtest, sin excepción**.

En los otros dos SKUs los tres modelos empatan dentro de ~1 punto porcentual, porque sus ventanas de
prueba tienen poca o nula actividad promocional.

**Decisión: un solo modelo (`lgbm_promo`) para los tres SKUs.** Promedia 11.2% de WAPE frente al 20.6%
que daría elegir el mejor modelo por SKU sin información promocional. Es a la vez más preciso y más
simple de operar que mantener tres modelos distintos — y evita el sobreajuste de "elegir el ganador
por SKU" sobre sólo 5 observaciones de backtest.

In [6]:
from forecasting_final import run as run_forecast
summary_a = run_forecast()
summary_a

,sku,wape_naive,wape_lgbm,wape_lgbm_promo,demanda_sem_hist_12s,forecast_sem_sin_promo,forecast_sem_con_promo_15pct
0,Shampoo Rizos 135 ml,0.162,0.139,0.143,958.9,959.1,1290.1
1,Desodorante 150 ml A,0.382,0.594,0.092,628.8,649.9,1360.4
2,Cubito de pollo c/50,0.100,0.104,0.102,981.9,1065.7,1155.0


### Forecast bajo dos escenarios

El forecast base **no asume promociones futuras**: proyecta demanda de negocio normal. Como el modelo
ahora entiende el efecto promocional, puede además simular el escenario "trimestre con combo al 15%",
que es la pregunta que el equipo comercial realmente quiere responder al planear reabasto.

La diferencia entre ambos escenarios es la señal de cuánto inventario adicional exige activar una promo:
para *Desodorante* la demanda semanal pasa de ~650 a ~1,360 unidades (**+109%**), consistente con el
uplift histórico de 86%–128% medido de forma independiente en el Reto C.

In [7]:
img = mpimg.imread(PROJECT_ROOT / "report/reto_a_forecasts.png")
plt.figure(figsize=(12, 13)); plt.imshow(img); plt.axis('off'); plt.show()

---
# Reto B — Sensibilidad al precio y simulador

**SKU:** *Antitranspirante 150 ml C* — el de mayor variación de precio observada (CV ≈ 0.10), lo que da
la mejor identificación disponible.

## Advertencia de identificación (leer antes de usar el número)

En este dataset el precio efectivo semanal **casi no varía por decisiones de pricing puras**: varía
porque hay o no un combo activo. La correlación entre `log(precio)` y la participación promocional de
la semana es **−0.93**.

Eso significa que el coeficiente estimado no es una elasticidad de precio de manual, sino el **efecto
combinado de activar un combo a profundidad *d***: precio + visibilidad en el punto de venta + mecánica
de bundle, todo junto. Es honesto nombrarlo así. También es útil: esa es exactamente la palanca que el
equipo comercial controla — no puede mover el precio de lista arbitrariamente, pero sí decide si lanza
un combo y a qué profundidad.

Para acotar cuánto depende el número de la especificación elegida, se ajustan tres.

In [8]:
from elasticity import ELASTICITY_SKU, price_panel, fit_specifications, run as run_elasticity

panel = price_panel(df, ELASTICITY_SKU)
print(f"{ELASTICITY_SKU}   semanas={len(panel)}   "
      f"rango de precio observado=[{panel.price.min():.2f}, {panel.price.max():.2f}]")
print(f"corr(log precio, promo_share) = {np.corrcoef(np.log(panel.price), panel.promo_share)[0,1]:.3f}")
print(f"precio medio  sin promo = {panel[panel.promo_share < .01].price.mean():.2f}   "
      f"con promo = {panel[panel.promo_share > .5].price.mean():.2f}")

specs = fit_specifications(panel)
for name, m in specs.items():
    c = 'log_price' if 'log_price' in m.params else 'discount'
    print(f"\n[{name}]\n    {c} = {m.params[c]:+.3f}   p = {m.pvalues[c]:.3f}   R² = {m.rsquared:.3f}")

Antitranspirante 150 ml C   semanas=104   rango de precio observado=[46.36, 63.39]
corr(log precio, promo_share) = -0.934
precio medio  sin promo = 60.33   con promo = 49.58

[base: log_price + tendencia + mes]
    log_price = -2.978   p = 0.000   R² = 0.945

[control por "hay combo activo"]
    log_price = -2.580   p = 0.017   R² = 0.945

[solo combo y profundidad (sin precio)]
    discount = +1.794   p = 0.302   R² = 0.940


**Robustez:** el coeficiente de precio se mueve entre **−2.98** y **−2.58** al agregar un control por
"hay combo activo", y sigue siendo significativo (p = 0.017). Es decir: el orden de magnitud es sólido
(demanda claramente elástica, en torno a −3), pero **reportar "−2.98" como si fuera un número preciso
sería falsa precisión**. La tercera especificación muestra el otro lado del problema: al quitar el precio
y dejar sólo `on_promo` + profundidad, ninguno de los dos resulta significativo por separado — están
demasiado correlacionados entre sí. Con esta data no se pueden separar.

## Simulador

`demanda(p) = demanda_ref × (p / precio_ref) ^ elasticidad`, anclado al promedio de las últimas 12
semanas, con costo y markup del SKU. La banda sombreada del gráfico refleja el rango de elasticidades
entre especificaciones.

**No se extrapola fuera del rango de precios observado** — un modelo log-log diverge rápido fuera de su
soporte y no hay evidencia que respalde esa zona.

In [9]:
r = run_elasticity()
print(f"Elasticidad usada: {r['elasticity']:.2f}  (rango entre especificaciones: "
      f"{min(r['elasticity_range']):.2f} a {max(r['elasticity_range']):.2f})")
print(f"Ancla: precio_ref = {r['ref_price']:.2f}, demanda_ref = {r['ref_qty']:.0f} u/sem, "
      f"costo = {r['econ']['unit_cost']:.2f}")

# ejemplo de uso: consultar el simulador en 3 precios
r['sim']([r['price_min'], r['ref_price'], r['price_max']]).round(2)

Elasticidad usada: -2.98  (rango entre especificaciones: -2.98 a -2.58)
Ancla: precio_ref = 63.34, demanda_ref = 366 u/sem, costo = 46.85


,price,demanda_esperada,ingreso,margen_abs,margen_pct,fuera_de_rango_observado
0,46.36,927.33,42994.40,-447.31,-0.01,False
1,63.34,366.25,23196.55,6039.14,0.26,False
2,63.39,365.36,23159.00,6043.39,0.26,False


In [10]:
g = r['grid']
print(f"Precio que maximiza MARGEN $ (en rango observado): {g.loc[g.margen_abs.idxmax(),'price']:.2f}")
print(f"Precio que maximiza INGRESO $:                     {g.loc[g.ingreso.idxmax(),'price']:.2f}")
print(f"Costo unitario:                                    {r['econ']['unit_cost']:.2f}")
print(f"Descuento de equilibrio del SKU:                   {r['econ']['breakeven_discount']:.1%}")

# ¿el óptimo teórico sin restricción de rango es siquiera alcanzable?
p_star = r['econ']['unit_cost'] * r['elasticity'] / (r['elasticity'] + 1)
print(f"\nÓptimo teórico de margen sin restricción: {p_star:.2f}  ->  "
      f"{'DENTRO' if r['price_min'] <= p_star <= r['price_max'] else 'FUERA'} del rango observado "
      f"[{r['price_min']:.2f}, {r['price_max']:.2f}]  (referencia, no recomendación)")

Precio que maximiza MARGEN $ (en rango observado): 63.39
Precio que maximiza INGRESO $:                     46.36
Costo unitario:                                    46.85
Descuento de equilibrio del SKU:                   23.1%

Óptimo teórico de margen sin restricción: 70.53  ->  FUERA del rango observado [46.36, 63.39]  (referencia, no recomendación)


In [11]:
img = mpimg.imread(PROJECT_ROOT / "report/reto_b_simulador.png")
plt.figure(figsize=(10, 6)); plt.imshow(img); plt.axis('off'); plt.show()

### Recomendación y riesgos

**El precio que maximiza ingreso (46.36) está por debajo del costo unitario (46.85).** Perseguir ingreso
en este SKU es literalmente vender a pérdida — es la tensión ingreso-vs-margen hecha número, y explica
por qué las promociones de este catálogo destruyen margen aun cuando "funcionan" en volumen.

Dentro del rango observado el margen se maximiza en el extremo superior (precio de lista). La
recomendación práctica: **no profundizar el descuento de este SKU**. Si el objetivo es volumen, conviene
una mecánica que preserve el precio unitario (bundle multi-SKU, regalo por compra, exhibición pagada)
antes que bajar precio.

**Riesgos y supuestos que limitan la confiabilidad:**

- **Identificación:** ya discutido — la variación de precio proviene casi enteramente de los combos. Esto
  es una relación observacional, no causal. Un test de precio aleatorizado por bodega la resolvería.
- **Elasticidad constante** es una simplificación; la sensibilidad puede cambiar de forma cerca del costo
  o cerca del precio de lista.
- **Colinealidad** entre estacionalidad y calendario promocional (los combos se lanzan en meses fijos):
  el modelo reparte ese efecto entre `log_price` y los dummies de mes de forma imprecisa.
- **El costo unitario sube ~5%/año** en este dataset; el simulador usa el costo reciente y no proyecta
  esa inflación hacia adelante.

---
# Reto C — Uplift promocional

**Método.** Para cada combo se ajusta un modelo estacional (tendencia + mes sobre `log(qty)`) entrenado
**únicamente con semanas sin promoción** de ese SKU, y se usa para predecir la demanda contrafactual
durante la ventana promocional. El uplift es la diferencia entre venta real y contrafactual.

Se prefirió esto a comparar contra el año anterior (varios combos no tienen equivalente) o contra
semanas adyacentes (que arrastran su propia estacionalidad).

Se calculó para **las 19 promociones**, no sólo dos, para poder elegir con evidencia comparativa.

## La métrica de decisión

El margen incremental se descompone de forma exacta:

$$\text{margen incremental} \;=\; \underbrace{I \cdot (P - C)}_{\text{ganancia por volumen}} \;-\; \underbrace{A_{promo} \cdot P \cdot d}_{\text{costo del descuento}}$$

donde $I$ = unidades incrementales, $A_{promo}$ = unidades vendidas en el combo, $P$ = precio de lista,
$C$ = costo unitario, $d$ = profundidad de descuento. De ahí sale el umbral de aprobación:

$$\frac{I}{A_{promo}} \;>\; \frac{P \cdot d}{P - C} \;=\; \frac{(1+m)\,d}{m}$$

Se reporta como **cobertura = uplift observado / uplift requerido**. Cobertura > 1 significa que la
promoción se pagó sola. Es comparable entre SKUs y traduce directo a una decisión.

> **Métrica descartada.** En una primera versión se ordenó el ranking por "margen incremental como % del
> ingreso incremental". Es una métrica rota: cuando el descuento es profundo y el uplift pequeño, el
> ingreso incremental puede ser **negativo**, y un margen negativo dividido entre un ingreso negativo da
> un porcentaje **positivo** — lo que colocaba las peores promociones en la cima del ranking. Se eliminó.

In [12]:
from uplift import PROMOS, estimate_uplift, run_all, plot_promos

detalle = plot_promos(df)
detalle.T

,0,1
sku,Antitranspirante 150 ml C,Desodorante 150 ml A
promo,Combo Verano 2,Combo Quincena Desodorante
inicio,2026-03-02,2025-08-04
fin,2026-05-10,2025-08-31
semanas,10,4
descuento,0.161,0.201
descuento_equilibrio,0.231,0.18
vende_bajo_costo,False,True
unidades_promo,8122,7738
unidades_reales,8122,7738


In [13]:
img = mpimg.imread(PROJECT_ROOT / "report/reto_c_uplift.png")
plt.figure(figsize=(12, 9)); plt.imshow(img); plt.axis('off'); plt.show()

### Validación cruzada de la métrica

El margen incremental se calcula por dos vías independientes: (a) empíricamente, con los montos
facturados reales de la ventana contra el contrafactual, y (b) analíticamente, con la descomposición
de arriba. Si el análisis está bien planteado, deben coincidir.

In [14]:
allc = run_all(df)
allc['descomposicion'] = allc.ganancia_por_volumen - allc.costo_del_descuento
allc['discrepancia_%'] = ((allc.margen_incremental - allc.descomposicion)
                          / allc.costo_del_descuento * 100).round(2)
print("Discrepancia máxima entre la vía empírica y la analítica: "
      f"{allc['discrepancia_%'].abs().max():.2f}%")
allc[['promo', 'margen_incremental', 'descomposicion', 'discrepancia_%']].head(8)

Discrepancia máxima entre la vía empírica y la analítica: 1.47%


,promo,margen_incremental,descomposicion,discrepancia_%
0,Combo Verano 2,-24643,-24779,0.17
1,Combo Verano Antitranspirante,-18961,-19189,0.43
2,Combo Quincena,-21998,-21623,-0.73
3,Combo Verano Desodorante,-45183,-45300,0.12
4,Combo Verano Desodorante 2,-69889,-69978,0.06
5,Combo Cierre Trimestre,-23578,-23558,-0.04
6,Combo Cabello Sano,-12567,-12587,0.10
7,Combo Cabello 3,-9548,-9589,0.30


Coinciden dentro de **1.5%**. La diferencia residual viene de que la vía analítica usa el descuento
promedio del combo mientras la empírica usa cada monto facturado.

*(Este chequeo detectó un error real: al usar el costo mediano histórico del SKU en vez del costo del
periodo, las dos vías divergían hasta 44%. El costo unitario sube ~5–6% al año en este dataset, así que
comparar promos de 2025 contra 2026 con un costo único sesga el resultado. Corregido anclando `P` y `C`
a la ventana de cada promoción.)*

### Canibalización

Un hecho estructural que condiciona toda la lectura: **durante la ventana de un combo, el 100% de la
venta de ese SKU es promocional**. No hay ventas orgánicas conviviendo con la promo.

In [15]:
print("Participación promocional dentro de la ventana de cada combo:")
print(f"   mínimo = {allc.share_promo_en_ventana.min():.0%}   "
      f"máximo = {allc.share_promo_en_ventana.max():.0%}")
print("\n-> La canibalización es total: toda unidad que se habría vendido igual")
print("   se vende con descuento. Por eso el 'costo del descuento' se aplica sobre")
print("   el volumen completo y no sólo sobre las unidades incrementales.")

Participación promocional dentro de la ventana de cada combo:
   mínimo = 100%   máximo = 100%

-> La canibalización es total: toda unidad que se habría vendido igual
   se vende con descuento. Por eso el 'costo del descuento' se aplica sobre
   el volumen completo y no sólo sobre las unidades incrementales.


### Las 19 promociones, ordenadas por cobertura

In [16]:
cols = ['sku', 'promo', 'descuento', 'descuento_equilibrio', 'vende_bajo_costo',
        'uplift_obs_pct', 'uplift_req_pct', 'cobertura', 'margen_incremental']
allc[cols]

,sku,promo,descuento,descuento_equilibrio,vende_bajo_costo,uplift_obs_pct,uplift_req_pct,cobertura,margen_incremental
0,Antitranspirante 150 ml C,Combo Verano 2,0.161,0.231,False,93.3,135.2,0.69,-24643
1,Antitranspirante 150 ml C,Combo Verano Antitranspirante,0.152,0.231,False,72.6,113.9,0.64,-18961
2,Antitranspirante 150 ml C,Combo Quincena,0.202,0.231,False,103.2,177.9,0.58,-21998
3,Desodorante 150 ml A,Combo Verano Desodorante,0.152,0.180,False,78.5,150.2,0.52,-45183
4,Desodorante 150 ml A,Combo Verano Desodorante 2,0.172,0.180,False,92.8,183.8,0.51,-69889
5,Antitranspirante 150 ml C,Combo Cierre Trimestre,0.220,0.231,False,87.8,179.3,0.49,-23578
6,Shampoo Rizos 135 ml,Combo Cabello Sano,0.152,0.213,False,38.8,99.1,0.39,-12567
7,Shampoo Rizos 135 ml,Combo Cabello 3,0.102,0.213,False,17.1,55.9,0.31,-9548
8,Shampoo Rizos 135 ml,Combo Regreso a Clases Cabello,0.122,0.213,False,21.0,69.2,0.30,-13355
9,Shampoo 135 ml Azul,Combo Azul Otoño,0.111,0.206,False,14.3,61.6,0.23,-5752


## Conclusiones del Reto C

**Ninguna de las 19 promociones fue rentable en margen** (cobertura máxima 0.69). No es un artefacto del
método: es aritmética de márgenes delgados. Con un markup de 24%, un descuento de 10% exige que las
unidades incrementales sean ~52% del volumen promocional para pagarse. Los uplifts observados rara vez
pasan de 25% en los SKUs de alimentos.

**Replicar — "Combo Verano 2" (Antitranspirante 150 ml C).** Mejor cobertura del catálogo (0.69): generó
+93% de unidades contra el +135% que necesitaba. Es la mecánica que más se acerca a pagarse sola. Con el
simulador del Reto B, bajar la profundidad de 16.1% a ~11% reduce el costo del descuento un tercio;
si el volumen aguanta —y la evidencia de "Combo Verano Antitranspirante" al 15.2% sugiere que sí—
queda cerca del punto de equilibrio. Vale probarla como test controlado, no como rollout completo.

**No replicar — "Combo Quincena Desodorante" y "Combo Cierre Trimestre Desodorante".** Descuentos de
20.1% y 21.1% sobre un SKU cuyo punto de equilibrio es 18.0%: **se vendió bajo costo**. Su cobertura es 0
por construcción — no existe volumen que las rescate. Y son especialmente engañosas porque *parecen*
exitosas: +86% y +128% de uplift, los números más altos del catálogo. Cuanto mejor funcionan, más dinero
cuestan.

Contraste que vale la pena señalar al equipo: *Combo Temporada Fría* (Cubito) tuvo el descuento más
suave del catálogo (10%) y aun así perdió $61,816, porque su uplift fue de apenas +2.2%. Descuento bajo
no equivale a promoción segura cuando no hay respuesta de demanda.

---
# Entregables

- **Documento de metodología, supuestos y trade-offs** (1–2 páginas): `report/VEMIO_metodologia_hallazgos.docx`
- **Recomendaciones de negocio** en lenguaje no técnico: sección 4 de ese documento.
- **Resultados intermedios** reproducibles en `data/`.